# FloodAI: Physics-Informed Flood Occurrence Prediction
**End-to-End Pipeline & Label Diagnostics for 3 Indian Basins**

This notebook runs the complete reproducible pipeline, including SoilGrids and Open-Elevation GIS joins, temporal splitting, resampling, Optuna XGBoost training, Leave-One-Basin-Out (LOBO) CV, and the Mahanadi label-rainfall coincidence diagnostic.

## Setup (Run once per session)

In [ ]:
# Mount Google Drive for persistent storage
from google.colab import drive
import os
drive.mount('/content/drive', force_remount=True)
os.makedirs('/content/drive/MyDrive/FloodAI_checkpoints', exist_ok=True)
print("Google Drive mounted.")

In [ ]:
# Clone repo and install requirements
import os
if not os.path.exists('/content/FloodAI'):
    !git clone https://github.com/souga15/FloodAI.git /content/FloodAI
else:
    !git -C /content/FloodAI pull origin main

os.chdir('/content/FloodAI')
!pip install -q -r requirements.txt
!pip install -q -e .
!pip install -q imdlib pysheds affine py3dep xarray shap optuna imbalanced-learn pyarrow
print("Environment setup complete.")

## Configuration Loading

In [ ]:
import sys
sys.path.insert(0, "src")

from pathlib import Path
from floodai.config import load_config
from floodai.logging_utils import setup_logging

CONFIG_PATH = Path("config/config.yaml")
cfg = load_config(CONFIG_PATH)
logger = setup_logging(cfg.output_dir, run_name=cfg.experiment_id)
print(f"Loaded experiment: {cfg.experiment_id}")

## Step 1 — Generate Basin Points (deterministic grid)

In [ ]:
from floodai.gis.points import generate_grid_fallback_points, basin_points_to_dataframe

all_points = []
for basin_key, basin_cfg in cfg.basins.items():
    if basin_key == "sutlej_punjab":
        continue  # Sutlej excluded from study scope
    pts = generate_grid_fallback_points(
        basin_key=basin_key,
        bbox=basin_cfg.bbox,
        n_points_target=basin_cfg.n_points_target,
        seed=cfg.random_seed,
    )
    all_points.extend(pts)

points_df = basin_points_to_dataframe(all_points)
print(f"Generated {len(points_df)} points across {points_df['basin_key'].nunique()} basins:")
print(points_df.groupby("basin_key").size())

## Step 2 — Collect Rainfall Data (IMD Gridded)

In [ ]:
import pandas as pd
from floodai.data.rainfall_providers import get_rainfall_provider

rainfall_cache_path = cfg.output_dir / "rainfall_raw.parquet"
if rainfall_cache_path.exists():
    print(f"[CACHE HIT] Loading rainfall from {rainfall_cache_path}")
    rainfall_df = pd.read_parquet(rainfall_cache_path)
else:
    print("Fetching rainfall data...")
    provider = get_rainfall_provider(cfg.raw["data_sources"]["rainfall"]["provider"])
    start_year = cfg.raw["data_sources"]["rainfall"]["start_year"]
    end_year = cfg.raw["data_sources"]["rainfall"]["end_year"]

    all_series = []
    for i, row in points_df.iterrows():
        try:
            df_point = provider.fetch_point_series(
                row["lat"], row["lon"], f"{start_year}0101", f"{end_year}1231"
            )
            df_point["point_id"] = row["point_id"]
            df_point["basin_key"] = row["basin_key"]
            all_series.append(df_point)
        except Exception as e:
            pass
        if (i + 1) % 20 == 0 or (i + 1) == len(points_df):
            print(f"  {i+1}/{len(points_df)} points fetched...")

    rainfall_df = pd.concat(all_series, ignore_index=True)
    cfg.output_dir.mkdir(parents=True, exist_ok=True)
    rainfall_df.to_parquet(rainfall_cache_path, index=False)
    print(f"Rainfall data saved to {rainfall_cache_path}")

## Step 3 — Load Verified Flood Events

In [ ]:
flood_events_path = Path(cfg.raw["data_sources"]["flood_events"]["path"])
flood_events_df = pd.read_csv(flood_events_path, parse_dates=["Start", "End"])
print(f"Loaded {len(flood_events_df)} verified events.")

## Step 4 — Feature Ingestion & Spatial Covariates (CN + TWI)

In [ ]:
from floodai.features.pipeline import (
    add_temporal_features, add_rainfall_window_features,
    add_scs_cn_runoff, add_interaction_features
)
from floodai.gis.terrain_real import add_real_terrain_features

# Temporal + rainfall windows
df = rainfall_df.merge(points_df[["point_id", "lat", "lon", "basin_key", "ESA_LandCover"]], on=["point_id", "basin_key"])
df = df.sort_values(["point_id", "Date"]).reset_index(drop=True)
df = add_temporal_features(df)
df = add_rainfall_window_features(df, group_col="point_id")

# Real terrain features (cached for speed)
terrain_cache_path = cfg.output_dir / "terrain_cache.parquet"
if terrain_cache_path.exists():
    print(f"[CACHE HIT] Restoring terrain from {terrain_cache_path}")
    terrain_df = pd.read_parquet(terrain_cache_path)
    df = df.merge(terrain_df, on=["point_id", "Date"], suffixes=("", "_dup"))
    df = df.loc[:, ~df.columns.str.endswith("_dup")]
else:
    print("Ingesting terrain features...")
    df = add_real_terrain_features(df, points_df, dem_cache_dir=str(cfg.output_dir / "dem_cache"))
    terrain_df = df[["point_id", "Date", "Elevation_m", "Curve_Number", "TWI"]]
    terrain_df.to_parquet(terrain_cache_path, index=False)

# SCS CN Runoff + Interaction terms
df = add_scs_cn_runoff(df)
df = add_interaction_features(df)
print(f"Feature matrix build complete. Shape: {df.shape}")

## Step 5 — Apply Flood Labels

In [ ]:
from floodai.features.labelling import label_floods
df = label_floods(df, flood_events_df)
print("Label distribution:")
print(df["Flood_Occurred"].value_counts())

## Step 6 — Hyperparameter Tuning & Model Fitting

In [ ]:
from floodai.features.governance import select_model_features
from floodai.training.split import split_train_val_test
from floodai.training.imbalance import resample_training_only
from floodai.training.tuning import run_optuna_search
from floodai.training.threshold import select_f1_optimal_threshold
from floodai.evaluation.metrics import evaluate, DataProvenance
import xgboost as xgb

feature_cols = select_model_features(df)
X_train, y_train, X_val, y_val, X_test, y_test = split_train_val_test(df, feature_cols, cfg.split)

X_train_res, y_train_res = resample_training_only(
    X_train, y_train,
    sampling_strategy=cfg.training["imbalance"]["sampling_strategy"],
    k_neighbors_max=cfg.training["imbalance"]["k_neighbors_max"],
    seed=cfg.random_seed,
)

best_params = run_optuna_search(
    X_train_res, y_train_res, X_val, y_val,
    n_trials=20, seed=cfg.random_seed
)

model = xgb.XGBClassifier(**best_params, random_state=cfg.random_seed)
model.fit(X_train_res, y_train_res)

y_pred_proba_val = model.predict_proba(X_val)[:, 1]
best_thresh = select_f1_optimal_threshold(y_val, y_pred_proba_val)

y_pred_proba_test = model.predict_proba(X_test)[:, 1]
res = evaluate(y_test, y_pred_proba_test, threshold=best_thresh, set_name="Test Set", provenance=DataProvenance.HELD_OUT)

print("\n" + "="*55)
print("  HELD-OUT TEST SET RESULTS")
print("="*55)
print(f"  ROC-AUC   : {res.roc_auc:.4f}")
print(f"  PR-AUC    : {res.pr_auc:.4f}")
print(f"  F1 Score  : {res.f1:.4f}")
print(f"  MCC       : {res.mcc:.4f}")
print(f"  FAR       : {res.far:.4f}")
print(f"  CSI       : {res.csi:.4f}")
print(f"  Threshold : {res.threshold:.4f}")
print("="*55)

## Step 7 — Leave-One-Basin-Out (LOBO) Validation

In [ ]:
from floodai.training.lobo import run_lobo_cv

lobo_results = run_lobo_cv(
    df, feature_columns=feature_cols, target_column='Flood_Occurred',
    basin_column='basin_key', best_params=best_params,
    early_stopping_rounds=10,
    smote_sampling_strategy=cfg.training["imbalance"]["sampling_strategy"],
    smote_k_neighbors_max=cfg.training["imbalance"]["k_neighbors_max"],
    seed=cfg.random_seed,
)

lobo_df = pd.DataFrame([{
    'held_out_basin': r.set_name.replace('LOBO_held_out_', ''),
    'ROC_AUC': r.roc_auc, 'PR_AUC': r.pr_auc, 'F1': r.f1, 'MCC': r.mcc,
    'n_test': r.n_samples, 'n_positive': r.n_positive,
} for r in lobo_results])

print("\n" + "="*65)
print("  LOBO CV RESULTS")
print("="*65)
print(lobo_df.to_string(index=False))
print("-"*65)
print(f"  Mean ROC-AUC : {lobo_df['ROC_AUC'].mean():.4f} ± {lobo_df['ROC_AUC'].std():.4f}")
print(f"  Mean F1      : {lobo_df['F1'].mean():.4f} ± {lobo_df['F1'].std():.4f}")
print("="*65)

## Step 8 — Mahanadi Label-Coincidence Diagnostic

In [ ]:
from floodai.evaluation.label_diagnostics import compare_basins_rainfall_coincidence
diag_df, cross_df = compare_basins_rainfall_coincidence(df, flood_events_df, baseline_years=[2017, 2024])

print("\n======================================================================")
print("  CROSS-BASIN SUMMARY")
print("======================================================================")
print(cross_df.to_string(index=False))